# 🎨 Udacity — Design of Computer Programs
### Peter Norvig


## 🃏 Lesson 1 — How to Write a Poker Program

### 🔍 Outlining the Problem

To write any program, follow 3 phases:
1. **Understand** — inventory the concepts of the problem
2. **Specify** — define it in terms that can become code
3. **Design** — structure it into code

---

### 📦 Concepts Inventory

| Concept | Description |
|---|---|
| **Card** | Has a `rank` (2–Ace) and a `suit` (♠ ♥ ♦ ♣) |
| **Hand** | A collection of 5 cards |
| **Hand Rank** | A value that determines how strong a hand is |
| **poker()** | Takes a list of hands → returns the best one |

---

### 🏆 Hand Rankings (strongest → weakest)

| # | Hand | Description |
|---|---|---|
| 1 | Straight Flush | 5 consecutive cards, same suit |
| 2 | Four of a Kind | 4 cards of the same rank |
| 3 | Full House | 3 of a kind + a pair |
| 4 | Flush | 5 cards of the same suit |
| 5 | Straight | 5 consecutive cards, any suit |
| 6 | Three of a Kind | 3 cards of the same rank |
| 7 | Two Pair | 2 different pairs |
| 8 | One Pair | 2 cards of the same rank |
| 9 | High Card | None of the above |

---

### 🧩 Key Concepts for Hand Rank Logic
- **n-of-a-kind** → n cards of the same rank (e.g. two 2s, two Jacks)
- **Straight** → 5 consecutive ranks (e.g. 5-6-7-8-9)
- **Flush** → 5 cards of the same suit (e.g. all ♠)
- **Ties** → when two hands have the same rank, compare by **card values** (higher card wins)

---

### 🎯 Goal of the `poker()` function

```python
def poker(hands):
    """Takes a list of hands, returns the best hand."""
    return max(hands, key=hand_rank)
```

> 💡 Using `max()` with a `key` function is a clean **functional programming** pattern — no loops needed.


### 🗂️ Representing Hands — Choosing a Data Structure

**Best choice → list of tuples** `[(rank, suit), ...]` ✅

| Option | Example | Why not? |
|---|---|---|
| List of strings | `['JS', 'JD', 'JC']` | Hard to extract rank/suit — need to parse the string |
| **List of tuples** ✅ | `[(11,'S'), (11,'D'), (11,'C')]` | Rank and suit already separated → easy to work with |
| Set of tuples | `{(11,'S'), (11,'D'), ...}` | Unordered — can't rely on card position |

```python
card = (11, 'S')   # Jack of Spades
card[0]            # → 11  (rank)
card[1]            # → 'S' (suit)
```


### 🎯 Poker Function & Testing

**Goal:** `poker(hands)` takes a list of hands and returns the best one.

```python
def poker(hands):
    "Return the best hand: poker([hand,...]) => hand"
    return max(hands, key=hand_rank)

def hand_rank(hand):
    return None  # placeholder — to be implemented later
```

> 💡 `max()` with `key=hand_rank` picks the hand with the highest rank value. Clean and no loops needed.

---

### ✅ Testing with `assert`

Write a `test()` function with **assert statements** — if an assertion is False, Python stops and raises an error immediately.

```python
def test():
    sf = "6C 7C 8C 9C TC".split()  # Straight Flush
    fk = "9D 9H 9S 9C 7D".split()  # Four of a Kind
    fh = "TD TC TH 7C 7D".split()  # Full House

    assert poker([sf, fk, fh]) == sf   # straight flush beats all
    assert poker([fk, fh]) == fk       # four of a kind beats full house
    assert poker([fh, fh]) == fh       # tie → returns first fh
```

**Why these test cases?**
- Test the **happy path** (sf wins)
- Test a **specific matchup** (fk vs fh)
- Test a **tie** — `max()` returns the first element when values are equal


### 🏅 hand_rank() — Returning a Comparable Value

**Goal:** `hand_rank(hand)` must return a value that allows comparing two hands correctly.

**Problem with returning just a single integer (0–8):**
```python
pair_of_10s → 1
pair_of_9s  → 1   # same rank! but 10s should win
```
→ A single number is **not enough** to break ties within the same rank.

---

### 🔢 Three Options to Fix This

| Option | Four-of-9s-with-5 example | Works? | Best? |
|---|---|---|---|
| Large integer | `70905` | ✅ | ❌ complicated arithmetic |
| Float | `7.0905` | ✅ | ❌ precision issues |
| **Tuple** | `(7, 9, 5)` | ✅ | ✅ simple, clean |

---

### ✅ Why Tuples Win

No arithmetic needed — just use commas:
```python
hand_rank(fk_9s) → (7, 9, 5)
#                   ↑  ↑  ↑
#                   │  │  └── kicker (remaining card)
#                   │  └───── rank of the 4-of-a-kind (9s)
#                   └──────── hand type: Four of a Kind = 7
```

---

### 📚 Lexicographic Ordering

Python compares tuples (and strings) **left to right** — this is called **lexicographic ordering**:

```python
(7, 9, 5) > (7, 3, 2)
# 7 == 7 → tie, move to next
# 9 > 3  → (7, 9, 5) wins ✅

'help' > 'hello'
# h==h, e==e, l==l → move on
# p > l → 'help' wins ✅
```

> 💡 Same principle used for sorting words in a dictionary — that's why it's called **lexico**graphic (lex = words).


### 🃏 hand_rank() — Full Tuple Structure per Hand

Each hand is described as a tuple: `(hand_type, tiebreakers...)`

| # | Hand | Example | Tuple returned |
|---|---|---|---|
| 8 | Straight Flush | J high | `(8, 11)` |
| 7 | Four of a Kind | Four Aces + Q kicker | `(7, 14, 12)` |
| 6 | Full House | Eights over Kings | `(6, 8, 13)` |
| 5 | Flush | 10, 8 high | `(5, 10, 8, ...)` all 5 cards |
| 4 | Straight | J high | `(4, 11)` |
| 3 | Three of a Kind | Three 7s | `(3, 7, 5, 2)` |
| 2 | Two Pair | Jacks and 3s | `(2, 11, 3, kicker)` |
| 1 | One Pair | Pair of 2s, J high | `(1, 2, 11, ...)` all cards |
| 0 | High Card | Nothing | `(0, ranks...)` all cards high→low |

**Key rules:**
- The **first number** = hand type (0–8)
- The **remaining numbers** = tiebreakers in order of importance
- For ties within the same hand type, Python compares the tuple **left to right** (lexicographic order)
- Cards use numeric ranks: `Jack=11, Queen=12, King=13, Ace=14`


### 🔧 Implementing hand_rank() with Tuples

`hand_rank()` returns a tuple where:
- **Element 0** → hand type (0–8)
- **Remaining elements** → tiebreakers in order of importance

```python
# Straight flush examples:
# ranks 2-3-4-5-6  → (8, 6)   ← high card is 6
# ranks 6-7-8-9-T  → (8, 10)  ← high card is 10
# (8, 10) > (8, 6) → second hand wins ✅
```

---

### 🧩 The `kind(n, ranks)` Function

`kind(n, ranks)` checks if there are **n cards of the same rank** and returns that rank (or `None`/falsy if not found).

```python
kind(4, ranks)  # → 9  if hand has four 9s  (truthy ✅)
kind(4, ranks)  # → None if no four of a kind (falsy ❌)
```

**Dual purpose — used as both:**
- A **value** → returns the rank (e.g. `9` for four 9s)
- A **boolean test** → truthy if found, falsy if not

```python
# In hand_rank():
if kind(4, ranks):              # ← boolean test
    return (7, kind(4, ranks),  # ← value usage
               kind(1, ranks))
```

> ⚠️ This works because card ranks go from **2 to 14** — `0` is never a valid rank, so returning `0` or `None` as a "not found" value is safe and always falsy in Python.

> 📝 **Note:** Unlike Java, Python treats `0`, `None`, and empty values as falsy — this dual-return pattern is a clean Python idiom.


### ✅ Testing hand_rank() — Adding Assertions per Hand

```python
sf = "6C 7C 8C 9C TC".split()  # Straight Flush  → high card = T (10)
fk = "9D 9H 9S 9C 7D".split()  # Four of a Kind  → four 9s, kicker 7
fh = "TD TC TH 7C 7D".split()  # Full House       → three 10s, pair of 7s

assert hand_rank(sf) == (8, 10)    # straight flush, 10 high
assert hand_rank(fk) == (7, 9, 7)  # four of a kind: four 9s + 7 kicker
assert hand_rank(fh) == (6, 10, 7) # full house: three 10s + pair of 7s
```

**How to read each tuple:**

| Hand | Tuple | Explanation |
|---|---|---|
| `sf` (6C 7C 8C 9C TC) | `(8, 10)` | Straight flush = 8, high card = T = 10 |
| `fk` (9D 9H 9S 9C 7D) | `(7, 9, 7)` | Four of a kind = 7, four 9s, kicker = 7D |
| `fh` (TD TC TH 7C 7D) | `(6, 10, 7)` | Full house = 6, three 10s, pair of 7s |

> 💡 Notice `fk` returns `(7, 9, 7)` — the first `7` is the **hand type** (four of a kind), and the second `7` is the **kicker card** (7 of diamonds). Two different 7s meaning different things!


### 🔧 hand_rank() — Complete Solution (All 9 Hand Types)

**Exercise:** fill in the return tuples for hands 6 → 0 using `kind()`, `flush()`, `straight()`, `two_pair()`, and `card_ranks()`.

```python
def hand_rank(hand):
    ranks = card_ranks(hand)
    if straight(ranks) and flush(hand):              # 8 — straight flush
        return (8, max(ranks))
    elif kind(4, ranks):                             # 7 — four of a kind
        return (7, kind(4, ranks), kind(1, ranks))
    elif kind(3, ranks) and kind(2, ranks):          # 6 — full house
        return (6, kind(3, ranks), kind(2, ranks))
    elif flush(hand):                                # 5 — flush
        return (5, ranks)
    elif straight(ranks):                            # 4 — straight
        return (4, max(ranks))
    elif kind(3, ranks):                             # 3 — three of a kind
        return (3, kind(3, ranks), ranks)
    elif two_pair(ranks):                            # 2 — two pair
        return (2, two_pair(ranks), ranks)
    elif kind(2, ranks):                             # 1 — one pair
        return (1, kind(2, ranks), ranks)
    else:                                            # 0 — high card
        return (0, ranks)
```

**Design decisions explained:**

| Hand | Tuple structure | Tiebreakers |
|---|---|---|
| Straight flush (8) | `(8, max(ranks))` | highest card |
| Four of a kind (7) | `(7, quad_rank, kicker)` | rank of the 4, then kicker |
| Full house (6) | `(6, triple_rank, pair_rank)` | rank of 3-of-a-kind first, then pair |
| Flush (5) | `(5, ranks)` | all 5 ranks in descending order |
| Straight (4) | `(4, max(ranks))` | highest card |
| Three of a kind (3) | `(3, triple_rank, ranks)` | rank of triple + remaining descending |
| Two pair (2) | `(2, (high_pair, low_pair), ranks)` | both pairs + kicker |
| One pair (1) | `(1, pair_rank, ranks)` | pair rank + remaining descending |
| High card (0) | `(0, ranks)` | all 5 ranks descending |

> 💡 `card_ranks()` returns ranks **sorted highest → lowest**, so when there is no special hand, just returning the full ranks list is enough — the highest card is automatically the first tiebreaker.

> 💡 For flush and high card, the whole `ranks` tuple is the tiebreaker — Python compares tuples element by element, so this correctly handles all edge cases automatically.


### 🃏 Helper Function: card_ranks()

**Purpose:** convert a hand (list of card strings) into a list of ranks **sorted highest → lowest**.

> 💡 Always sorted descending because ranks are used as tiebreakers — the highest card must come first.

**New test assertions added to `test()`:**

```python
assert card_ranks(sf) == [10, 9, 8, 7, 6]   # straight flush: T=10, descending
assert card_ranks(fk) == [9, 9, 9, 9, 7]    # four 9s first, then kicker 7
assert card_ranks(fh) == [10, 10, 10, 7, 7]  # three 10s first, then pair of 7s
```

| Hand | Cards | card_ranks output | Why? |
|---|---|---|---|
| `sf` | 6C 7C 8C 9C TC | `[10, 9, 8, 7, 6]` | T=10, sorted high→low |
| `fk` | 9D 9H 9S 9C 7D | `[9, 9, 9, 9, 7]` | four 9s before the 7 kicker |
| `fh` | TD TC TH 7C 7D | `[10, 10, 10, 7, 7]` | three 10s before the pair of 7s |

> 💡 `card_ranks()` only cares about **rank**, not suit. Face cards map to integers: J=11, Q=12, K=13, A=14, T=10.


### 🔧 Implementing card_ranks() — Mapping Face Cards to Integers

**Problem:** pulling the rank character directly gives wrong ordering — `'T'` sorts alphabetically *after* `'A'`, `'Q'`, etc., which is incorrect.

**Solution:** use `.index()` on a reference string where position = value:

```python
def card_ranks(cards):
    "Return a list of the ranks, sorted with higher first."
    ranks = ['--23456789TJQKA'.index(r) for r, s in cards]
    ranks.sort(reverse=True)
    return ranks

# card_ranks(['AC', '3D', '4S', 'KH']) => [14, 13, 4, 3]
```

**How the index trick works:**

```
'--23456789TJQKA'
  0123456789...
         ↑ T is at index 10 → maps to 10
              ↑ J = 11, Q = 12, K = 13, A = 14
```

| Character | Index in string | Numeric rank |
|---|---|---|
| `'2'` | 2 | 2 |
| `'9'` | 9 | 9 |
| `'T'` | 10 | 10 |
| `'J'` | 11 | 11 |
| `'Q'` | 12 | 12 |
| `'K'` | 13 | 13 |
| `'A'` | 14 | 14 |

> 💡 The `--` at position 0 and 1 are placeholder dashes — no card has rank 0 or 1. This keeps the indexing aligned so `'2'` lands at index 2 naturally.

> 💡 Alternative approaches (dict lookup, if-elif chain) would work too, but the index string is the most concise.


### 🔀 Helper Functions: straight() and flush()

**Tests added first:**

```python
assert straight([9, 8, 7, 6, 5]) == True   # consecutive ranks → straight
assert straight([9, 8, 8, 6, 5]) == False  # pair: not a straight
assert flush(sf) == True                    # sf = all clubs → flush
assert flush(fk) == False                   # fk = 4 different suits → not a flush
```

**Solutions:**

```python
def straight(ranks):
    "Return True if the ordered ranks form a 5-card straight."
    return (max(ranks) - min(ranks) == 4) and (len(set(ranks)) == 5)

def flush(hand):
    "Return True if all the cards have the same suit."
    suits = [s for r, s in hand]
    return len(set(suits)) == 1
```

**straight() logic:**

| Condition | Why? |
|---|---|
| `max - min == 4` | 5 consecutive cards span exactly 4 (e.g. 5→9: 9−5=4) |
| `len(set(ranks)) == 5` | all 5 ranks are different (rules out pairs like [9,8,8,6,5]) |

> 💡 Both conditions together are enough — 5 distinct ranks spanning 4 = guaranteed consecutive sequence.

**flush() logic:**

- Extract suits: `[s for r, s in hand]` — each card is a 2-char string, `s` is the second character
- `set(suits)` collapses duplicates → if only 1 unique suit remains, all cards share the same suit


### 🔧 Helper Function: kind(n, ranks)

**Goal:** return the first rank that appears exactly `n` times in the hand. Return `None` if not found.

```python
def kind(n, ranks):
    """Return the first rank that this hand has exactly n of.
    Return None if there is no n-of-a-kind in the hand."""
    for r in ranks:
        if ranks.count(r) == n:
            return r
    return None
```

**How it works:**
- Loop through each rank in the (sorted) list
- `.count(r)` counts how many times `r` appears in the list
- If it matches exactly `n` → return that rank
- If nothing matches → return `None` (falsy)

---

**Test assertions:**

```python
tp = "5S 5D 9H 9C 6S".split()  # Two Pair: pair of 9s and pair of 5s, kicker 6
fkranks = card_ranks(fk)        # [9, 9, 9, 9, 7]
tpranks = card_ranks(tp)        # [9, 9, 6, 5, 5]

assert kind(4, fkranks) == 9    # four 9s → returns 9
assert kind(3, fkranks) == None # no three of a kind (it's four!) → None
assert kind(2, fkranks) == None # no pair → None
assert kind(1, fkranks) == 7    # one 7 (the kicker) → returns 7
```

**Key insight — exactly n, not at least n:**

| Call | fkranks = [9,9,9,9,7] | Why? |
|---|---|---|
| `kind(4, fkranks)` | `9` | four 9s → exactly 4 ✅ |
| `kind(3, fkranks)` | `None` | 9 appears 4 times, not 3 ❌ |
| `kind(2, fkranks)` | `None` | no rank appears exactly 2 times ❌ |
| `kind(1, fkranks)` | `7` | 7 appears exactly once ✅ |

> 💡 `kind()` uses `.count()` — a built-in list method that counts occurrences of an element.
> Since ranks are sorted **highest → lowest**, it always returns the **highest** matching rank first.


### 🔧 Helper Function: two_pair(ranks)

**Goal:** if there are two pairs, return `(highest_pair, lowest_pair)`; otherwise `None`.

```python
def two_pair(ranks):
    """If there are two pair, return the two ranks as a
    tuple: (highest, lowest); otherwise return None."""
    pair  = kind(2, ranks)           # highest pair (ranks sorted high→low)
    lowpair = kind(2, ranks[::-1])   # lowest pair (scan from the end)
    if pair and lowpair and pair != lowpair:
        return (pair, lowpair)
    return None
```

**How it works:**

| Step | Code | Result for `tpranks = [10, 10, 9, 7, 7]`* |
|---|---|---|
| Find highest pair | `kind(2, ranks)` | `10` — first pair found left→right |
| Find lowest pair | `kind(2, ranks[::-1])` | `7` — first pair found right→left |
| Check they differ | `pair != lowpair` | `10 != 7` ✅ → return `(10, 7)` |
| Single pair case | `pair == lowpair` | same rank = only 1 pair → `None` |

> 💡 `ranks[::-1]` reverses the list — since ranks are sorted **high → low**, reversing gives **low → high**, so `kind()` finds the lowest pair first.

> 💡 The check `pair != lowpair` is necessary to avoid returning `(9, 9)` when there's a three-of-a-kind — `kind(2, ...)` would find the same rank from both ends.

---

### ✅ Lesson 1 — Complete Program Summary

All helper functions implemented and tested:

```python
def card_ranks(hand):   # rank chars → sorted ints [14..2]
def straight(ranks):    # max-min==4 and 5 unique ranks
def flush(hand):        # all suits identical
def kind(n, ranks):     # first rank appearing exactly n times
def two_pair(ranks):    # (high_pair, low_pair) or None
def hand_rank(hand):    # returns comparable tuple (0–8, tiebreakers...)
def poker(hands):       # max(hands, key=hand_rank)
```

> 💡 Norvig closes Lesson 1 noting the test suite is not exhaustive yet — in the next videos he'll show an edge case that breaks the program and introduce more rigorous testing strategies.


---

## 🐛 Edge Case: Ace-Low Straight (A-2-3-4-5)

The Ace can count as **low** only in `A-2-3-4-5` (the "wheel"). Our program breaks this in 2 ways:
- `ranks = [14,5,4,3,2]` → max-min = 12 ≠ 4 → **not detected as straight**
- Even if detected, `max(ranks) = 14` → **wrong high card** (should be 5)

**Naïve fix:** change `card_ranks()`, `straight()`, and `hand_rank()`.

### 💡 Norvig's Design Principle
> *"The amount of change in code should be proportional to the amount of change in the concept."*

One conceptual change → one code change. Fix it **only in `card_ranks()`**: replace `[14,5,4,3,2]` with `[5,4,3,2,1]`. The other functions need zero changes.


### ✅ Fix: Ace-Low Straight in card_ranks()

Only `card_ranks()` needs to change — one special-case check at the end:

```python
def card_ranks(hand):
    "Return a list of the ranks, sorted with higher first."
    ranks = ['--23456789TJQKA'.index(r) for r, s in hand]
    ranks.sort(reverse=True)
    return [5, 4, 3, 2, 1] if ranks == [14, 5, 4, 3, 2] else ranks
```

**Why this works:**
- `[14, 5, 4, 3, 2]` is the only case where Ace plays low
- Replaced with `[5, 4, 3, 2, 1]` → `max-min = 5-1 = 4` ✅ and 5 unique values ✅
- `straight()` and `hand_rank()` need **zero changes**

```python
assert straight(card_ranks("AC 2D 4H 3D 5S".split())) == True  ✅
```

> 💡 `1` is only ever returned in this single edge case — all other ranks are 2–14.
